## Education Analytics Prediction Platform

### Feature Engineering

\Importing Necessary Libraries

In [54]:
import numpy as np
import pandas as pd
import sys
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.utils.class_weight import compute_class_weight

sys.path.insert(0, "../src")
from data_cleaning import clean_dataset

import warnings
warnings.filterwarnings("ignore")

df_raw = pd.read_csv(r"C:\NG\Educational-Analytics-Platform\data\raw\student_dropout_dataset.csv")
target = "Dropout"
df, _ = clean_dataset(df_raw, target=target)

print(f"Shape: {df.shape}")
df.head(3)

2026-07-01 17:06:40,069 | data_cleaning | INFO | Starting cleaning pipeline. Shape: (10000, 19)
2026-07-01 17:06:40,180 | data_cleaning | INFO | Removed 0 duplicate rows
2026-07-01 17:06:40,303 | data_cleaning | INFO | Missing value detection: 4 columns have missing values
2026-07-01 17:06:40,354 | data_cleaning | INFO | Missing treatment done: 4 filled, 0 dropped, 0 remaining
2026-07-01 17:06:40,405 | data_cleaning | INFO | Outlier detection: 6 columns have outliers
2026-07-01 17:06:40,422 | data_cleaning | INFO | Outlier treatment: capped values in 6 columns
2026-07-01 17:06:40,426 | data_cleaning | INFO | Cleaning pipeline complete. Final shape: (10000, 19)


Shape: (10000, 19)


,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0


\Datetime Features

In [37]:
def engineer_datetime_features(df):
    
    df_new = df.copy()
    created_features = []

    datetime_cols = []
    for col in df_new.columns:
        if df_new[col].dtype == "datetime64[ns]":
            datetime_cols.append(col)
        elif df_new[col].dtype == "object":
            parsed = pd.to_datetime(df_new[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                datetime_cols.append(col)

    for col in datetime_cols:
        dt = pd.to_datetime(df_new[col], errors="coerce")
        df_new[f"{col}_year"] = dt.dt.year
        df_new[f"{col}_month"] = dt.dt.month
        df_new[f"{col}_quarter"] = dt.dt.quarter
        df_new[f"{col}_day"] = dt.dt.day
        df_new[f"{col}_weekday"] = dt.dt.weekday
        created_features.extend([f"{col}_year", f"{col}_month",
                                 f"{col}_quarter", f"{col}_day", f"{col}_weekday"])
        
        df_new = df_new.drop(columns=[col])

    if created_features:
        print(f"Created {len(created_features)} datetime features from {len(datetime_cols)} column(s):")
        print(f"  {created_features}")
    else:
        print("No datetime columns found. Skipping datetime feature engineering.")

    return df_new, created_features

In [38]:
df, dt_features = engineer_datetime_features(df)
print(f"Shape after: {df.shape}")

No datetime columns found. Skipping datetime feature engineering.
Shape after: (10000, 19)


\dropping useless columns

In [39]:
def drop_useless_columns(df, target=None):
    df_new = df.copy()
    dropped = []

    for col in df.columns:
        if col == target:
            continue
        if df_new[col].nunique() == len(df_new):
            df_new = df_new.drop(columns=[col])
            dropped.append(col)

    if dropped:
        print(f"Dropped ID-like columns: {dropped}")
    else:
        print("No ID-like columns to drop.")

    return df_new, dropped

In [40]:
df,dropped_ids = drop_useless_columns(df,target=target)
print(f"Shape after: {df.shape}")

Dropped ID-like columns: ['Student_ID']
Shape after: (10000, 18)


\Feature Encoding

In [41]:
def encode_features(df,target=None,low_card_max=10):
    df_new = df.copy()
    encoding_map = {"Binary" : {}, "One-Hot" : [] , "frequency" : {}}

    cat_cols = [c for c in df_new.select_dtypes(include = "object").columns if c!= target]

    for col in cat_cols:
        n_unique = df_new[col].nunique()

        #Binary
        if n_unique == 2:
            le = LabelEncoder()
            df_new[col] = le.fit_transform(df_new[col])
            encoding_map["Binary"][col] = dict(zip(le.classes_,le.transform(le.classes_)))

        #low-cardinality
        elif 3 <= n_unique < low_card_max:
            dummies = pd.get_dummies(df_new[col],prefix = col, drop_first = True)
            dummies = dummies.astype(int)
            df_new = pd.concat([df_new.drop(columns=[col]),dummies],axis = 1)
            encoding_map["One-Hot"].append(col)

        #High-cardinality
        else:
            freq = df_new[col].value_counts().to_dict()
            df_new[col] = df_new[col].map(freq)
            encoding_map["frequency"][col] = freq

    print(f"Binary Encoded : {list(encoding_map['Binary'].keys())}")
    print(f"One-Hot Encoded : {encoding_map['One-Hot']}")
    print(f"Frequency Encoded : {list(encoding_map['frequency'].keys())}")

    return df_new, encoding_map

In [42]:
df,encoding_map = encode_features(df, target=target)
print(f"Shape after encoding: {df.shape}")
df.head(3)

Binary Encoded : ['Gender', 'Internet_Access', 'Part_Time_Job', 'Scholarship']
One-Hot Encoded : ['Semester', 'Department', 'Parental_Education']
Frequency Encoded : []
Shape after encoding: (10000, 25)


,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,...,Semester_Year 2,Semester_Year 3,Semester_Year 4,Department_Business,Department_CS,Department_Engineering,Department_Science,Parental_Education_High School,Parental_Education_Master,Parental_Education_PhD
0,22.1,1,25000.0,1,3.36,86.1,2,20.4,1,0,...,0,0,0,0,0,0,0,1,0,0
1,20.7,1,25000.0,1,4.30,68.0,2,44.0,0,0,...,0,1,0,0,0,1,0,0,0,0
2,22.4,1,40183.0,1,4.40,70.9,0,48.9,1,0,...,0,0,0,0,0,0,0,0,1,0


In [43]:
def scale_features(df, target=None, min_unique=10):
    
    df_new = df.copy()

    numeric_cols = df_new.select_dtypes(include=np.number).columns.tolist()
    continuous_cols = [
        c for c in numeric_cols
        if c != target and df_new[c].nunique() >= min_unique
    ]

    if len(continuous_cols) == 0:
        print("No continuous numeric columns to scale.")
        return df_new, None, []

    scaler = StandardScaler()
    df_new[continuous_cols] = scaler.fit_transform(df_new[continuous_cols])

    print(f"Scaled {len(continuous_cols)} continuous columns:")
    print(f"  {continuous_cols}")
    return df_new, scaler, continuous_cols

df, scaler, scaled_cols = scale_features(df, target=target)
df.head(3)

## Feature Selection

\remove constant feature

In [44]:
def remove_constant_features(df,target=None,threshold=0.99):

    df_new = df.copy()
    dropped = []
    for col in df_new.columns:
        if col == target:
            continue

        top_ratio = df_new[col].value_counts(normalize=True).values[0]
        if top_ratio >=threshold:
            df_new = df_new.drop(columns=[col])
            dropped.append((col,round(top_ratio,3)))
    return df_new , dropped

\remove correlated feature

In [45]:
def remove_correlated_features(df,target=None,threshold=0.90):

    df_new = df.copy()
    numeric_cols = [c for c in df_new.select_dtypes(include=np.number).columns if c != target]

    corr = df_new[numeric_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    dropped = [col for col in upper.columns if any(upper[col] >= threshold)]
    df_new = df_new.drop(columns=dropped)
    return df_new,dropped

In [49]:
def select_by_mutual_info(df,target,k=None,min_mi=0.001):

    df_new = df.copy()
    X = df_new.drop(columns=[target])
    y = df_new[target]

    mi_scores = mutual_info_classif(X,y,random_state=42)
    mi_series = pd.Series(mi_scores, index = X.columns).sort_values(ascending=False)

    low_mi = mi_series[mi_series < min_mi].index.tolist()
    df_new = df_new.drop(columns=low_mi)

    return df_new,mi_series,low_mi

\Feature selection

In [47]:
def select_features(df, target):
    
    report = {"start_features": df.shape[1] - 1}

    df, constant_dropped = remove_constant_features(df, target)
    report["constant_dropped"] = constant_dropped

    df, corr_dropped = remove_correlated_features(df, target)
    report["correlated_dropped"] = corr_dropped

    df, mi_series, low_mi_dropped = select_by_mutual_info(df, target)
    report["low_mi_dropped"] = low_mi_dropped
    report["mi_scores"] = mi_series.round(4).to_dict()
    report["final_features"] = df.shape[1] - 1

    return df, report

In [52]:
df,feature_report = select_features(df,target)
feature_report

{'start_features': 16,
 'constant_dropped': [],
 'correlated_dropped': [],
 'low_mi_dropped': ['Assignment_Delay_Days',
  'Gender',
  'Part_Time_Job',
  'Semester_Year 3',
  'Department_CS',
  'Semester_Year 4'],
 'mi_scores': {'GPA': 0.1141,
  'Stress_Index': 0.032,
  'Study_Hours_per_Day': 0.0141,
  'Attendance_Rate': 0.0078,
  'Internet_Access': 0.0064,
  'Semester_Year 2': 0.0054,
  'Family_Income': 0.0048,
  'Scholarship': 0.004,
  'Parental_Education_PhD': 0.0031,
  'Parental_Education_Master': 0.0026,
  'Assignment_Delay_Days': 0.0003,
  'Gender': 0.0,
  'Part_Time_Job': 0.0,
  'Semester_Year 3': 0.0,
  'Department_CS': 0.0,
  'Semester_Year 4': 0.0},
 'final_features': 10}

\Class Imbalance Detect

In [55]:
def detect_imbalance(df, target, threshold=1.5):
    
    counts = df[target].value_counts()
    pct = df[target].value_counts(normalize=True) * 100

    imbalance_ratio = counts.max() / counts.min()
    is_imbalanced = imbalance_ratio > threshold

    classes = np.sort(df[target].unique())
    weights = compute_class_weight(class_weight="balanced",
                                   classes=classes, y=df[target])
    class_weights = dict(zip(classes, weights.round(3)))

    report = {
        "counts": counts.to_dict(),
        "percentage": pct.round(2).to_dict(),
        "imbalance_ratio": round(imbalance_ratio, 2),
        "is_imbalanced": is_imbalanced,
        "class_weights": class_weights,
        "strategy": "class_weight='balanced'" if is_imbalanced else "none needed",
    }
    return report

In [56]:
def print_imbalance_report(report):
    print("=" * 50)
    print("  CLASS IMBALANCE REPORT")
    print("=" * 50)
    print("Class distribution:")
    for cls, count in report["counts"].items():
        pct = report["percentage"][cls]
        print(f"  Class {cls}: {count} ({pct}%)")
    print(f"\nImbalance ratio : {report['imbalance_ratio']}")
    print(f"Is imbalanced   : {report['is_imbalanced']}")
    print(f"Class weights   : {report['class_weights']}")
    print(f"Strategy        : {report['strategy']}")


In [57]:
imb_report = detect_imbalance(df, target)
print_imbalance_report(imb_report)

  CLASS IMBALANCE REPORT
Class distribution:
  Class 0: 7646 (76.46%)
  Class 1: 2354 (23.54%)

Imbalance ratio : 3.25
Is imbalanced   : True
Class weights   : {np.int64(0): np.float64(0.654), np.int64(1): np.float64(2.124)}
Strategy        : class_weight='balanced'
